# Capital Region First Ring Expressway (formerly as Seoul Outer Ring Expressway) — 3D label (spatial × temporal) regression demo

This notebook shows a spatio-temporal regression workflow where inputs X are (month, day-type, direction) and labels Y are a (segment × hour) grid.
- Features X: shape (B, D) where B is the number of (month, day-type, direction) groups; D is feature dim (e.g., 3).
- Labels Y: shape (B, S, T) where S is segment count, T is 24 hours.
- Model: `stnet` spatiotemporal model built with `Model` + `ModelConfig/PatchConfig`.
- Runtime: works on Colab T4 (CUDA 12.x) or CPU; adjusts config per device.
- Note: this is a shape/alignment demo; tweak hyperparameters for real training.


## 1. Environment and dependencies
Install the analysis stack (tensordict, h5py included). Triton ships with PyTorch, so we do not force-install it.


In [ ]:
# Colab/local common dependencies (assume stnet is installed separately)
import sys, importlib


def _ensure(pkg: str, mod: str | None = None) -> None:
    mod = mod or pkg
    try:
        importlib.import_module(mod)
    except Exception:
        import subprocess

        print(f"[install] {pkg} ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pkg],
            check=True,
        )


# Core analytics stack (h5py/tensordict included; Triton ships with PyTorch)
_ensure("polars")
_ensure("tensordict")
_ensure("h5py")
_ensure("fastexcel")
_ensure("xlsxwriter")
_ensure("matplotlib")
_ensure("openpyxl")
_ensure("pandas")

import os, math
from typing import Any, Dict, List, Tuple, Optional
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.dpi"] = 120

# Detect Colab and mount Google Drive
try:
    import google.colab  # type: ignore

    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=True)
        print("[info] Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"[warn] Google Drive mount failed: {e!r}")

print(f"IS_COLAB = {IS_COLAB}")

In [ ]:
import torch, importlib, sys


def _ensure(pkg: str, mod: str | None = None) -> None:
    mod = mod or pkg
    try:
        importlib.import_module(mod)
    except Exception:
        import subprocess

        print(f"[install] {pkg} ...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pkg],
            check=True,
        )


# GPU-only packages: try installing only on CUDA 12.x (Colab T4 uses cu128)
if torch.cuda.is_available():
    cuda_ver = (torch.version.cuda or "").strip()
    if cuda_ver.startswith("12"):
        _ensure("transformer-engine-cu12", mod="transformer_engine")
        _ensure("transformer-engine-torch")
    else:
        print(f"[skip] transformer-engine (CUDA {cuda_ver})")
else:
    print("[skip] GPU-only extras (no CUDA visible)")

## 2. Input Excel path and output directory
Defaults to the bundled `raw_data.xlsx`. On Colab, if `/content/drive/My Drive/raw_data.xlsx` exists, it is used; otherwise we warn and fall back. Working directory is set to the file's parent.


In [ ]:
from typing import Union
import os
from pathlib import Path

ROOT = Path.cwd()
DEFAULT_RAW = ROOT / "raw_data.xlsx"
EXCEL_PATH = DEFAULT_RAW

if IS_COLAB:
    drive_candidate = Path("/content/drive/My Drive/raw_data.xlsx")
    if drive_candidate.exists():
        EXCEL_PATH = drive_candidate
    else:
        print(
            f"[warn] Drive file missing: {drive_candidate}; using bundled raw_data.xlsx"
        )

OUT_DIR = EXCEL_PATH.parent
os.makedirs(OUT_DIR, exist_ok=True)
os.chdir(OUT_DIR)

print("EXCEL_PATH:", EXCEL_PATH)
print("OUT_DIR   :", str(OUT_DIR))

## 3. Convert wide Excel to long format
Each sheet name encodes (month, weekday). We load every sheet, add route info, and unpivot hour columns into a long table (route, segment, direction, month, weekday, hour, value).


In [ ]:
# Each Excel sheet contains columns like '노선', '구간', '방향', '00시'~'23시'
import re
import polars as pl
import pandas as pd
from openpyxl import load_workbook
from IPython.display import display


def parse_sheet_name(name: str) -> tuple[int, str]:
    """Extract (month, weekday) from sheet name. Example: '01월 월요일' -> (1, '월요일')"""

    # 1) Extract month
    m = re.search(r"(\d+)월", name)
    if not m:
        raise ValueError(f"Could not find month in sheet name: {name}")
    month = int(m.group(1))

    # 2) Extract weekday (avoid confusing the '월' character inside the month substring)
    name_clean = name.replace(m.group(0), "")

    if "월" in name_clean:
        day_kind = "월요일"
    elif "화" in name_clean:
        day_kind = "화요일"
    elif "수" in name_clean:
        day_kind = "수요일"
    elif "목" in name_clean:
        day_kind = "목요일"
    elif "금" in name_clean:
        day_kind = "금요일"
    elif "토" in name_clean:
        day_kind = "토요일"
    elif "일" in name_clean:
        day_kind = "일요일"
    else:
        raise ValueError(f"Could not find weekday in sheet name: {name}")

    return month, day_kind


def _read_excel_polars(path, sheet_name: str) -> pl.DataFrame:
    """Best-effort Excel -> Polars reader.

    - Prefer pl.read_excel (fast when fastexcel is available)
    - Fallback to pandas.read_excel -> pl.from_pandas when needed
    """
    try:
        return pl.read_excel(source=str(path), sheet_name=sheet_name)
    except Exception as e:
        print(
            f"[warn] pl.read_excel failed (sheet={sheet_name}): {e!r} -> fallback pandas"
        )
        df_pd = pd.read_excel(path, sheet_name=sheet_name)
        return pl.from_pandas(df_pd)


HOURS = [f"{h:02d}시" for h in range(24)]

# Sheet list (read-only, fast metadata access)
workbook = load_workbook(EXCEL_PATH, read_only=True, data_only=True)
sheet_names: list[str] = list(workbook.sheetnames)
workbook.close()

print(f"Total sheets: {len(sheet_names)}")
print("Sample sheets:", sheet_names[:8])

# --- Wide -> long (LAZY) ---
# NOTE: Excel IO itself isn't lazy (Excel needs to be read), but everything after read_excel()
# is kept as a LazyFrame pipeline until we explicitly collect().
long_parts_lf: list[pl.LazyFrame] = []

for sh in sheet_names:
    try:
        df_pl = _read_excel_polars(EXCEL_PATH, sh)
    except Exception as e:
        print(f"[warn] skip sheet={sh}: {e!r}")
        continue

    # Normalize column names
    df_pl = df_pl.rename({c: str(c).strip() for c in df_pl.columns})

    # Required columns
    if not {"구간", "방향"}.issubset(df_pl.columns):
        print(f"[warn] skipped: required columns missing (sheet={sh})")
        continue

    hour_cols = [c for c in df_pl.columns if c in HOURS]
    if not hour_cols:
        print(f"[warn] skipped: hour columns missing (sheet={sh})")
        continue

    month, day_kind = parse_sheet_name(sh)

    # Ensure '노선' exists (some Excel exports omit it)
    if "노선" not in df_pl.columns:
        df_pl = df_pl.with_columns(pl.lit("수도권제1순환선").alias("노선"))

    # Preserve original row order inside each sheet (for debugging / row alignment)
    df_pl = df_pl.with_row_index(name="row_in_sheet", offset=0)

    cols_to_select = ["row_in_sheet", "노선", "구간", "방향"] + hour_cols

    lf = (
        df_pl.select(cols_to_select)
        .lazy()
        .with_columns(
            [
                pl.lit(month).alias("월"),
                pl.lit(day_kind).alias("일종"),
            ]
        )
        .unpivot(
            index=["row_in_sheet", "노선", "구간", "방향", "월", "일종"],
            on=hour_cols,
            variable_name="시간",
            value_name="지표",
        )
    )
    long_parts_lf.append(lf)

assert long_parts_lf, "Found no valid sheets to process."

long_lf = pl.concat(long_parts_lf, how="vertical")

# Type cleanup (still lazy)
long_lf = long_lf.with_columns(
    [
        pl.col("시간")
        .cast(pl.Utf8)
        .str.replace("시", "")
        .cast(pl.Int64, strict=False),
        pl.col("지표").cast(pl.Float64, strict=False),
    ]
).drop_nulls("지표")

print("long_lf (lazy plan):")
print(long_lf)

# Preview only (do NOT collect full dataset yet)
display(long_lf.limit(10).collect())

## 4. Encode IDs and seg_idx
Map weekday/direction to integer IDs and assign `seg_idx` per road segment. Keep `long_df` with columns like [month, weekday_id, direction_id, seg_idx, hour, value].


In [ ]:
# Encode IDs and seg_idx (keep the pipeline lazy as long as possible)

# 1) Encode weekday and direction IDs (7 weekdays)
DAY2ID: dict[str, int] = {
    "월요일": 0,
    "화요일": 1,
    "수요일": 2,
    "목요일": 3,
    "금요일": 4,
    "토요일": 5,
    "일요일": 6,
}
DIR2ID: dict[str, int] = {"상행": 0, "하행": 1}

# Apply mappings (LAZY)
long_lf = long_lf.with_columns(
    [
        pl.col("일종").replace(DAY2ID).cast(pl.Int64).alias("요일타입_id"),
        pl.col("방향").replace(DIR2ID).cast(pl.Int64).alias("방향_id"),
    ]
)

# 2) Canonicalize section names: treat 'A-B' and 'B-A' as the same physical segment
# (pure Polars expression; no Python UDF -> stays optimizable in lazy mode)
canonical_section_expr = (
    pl.col("구간")
    .cast(pl.Utf8)
    .str.split("-")
    .list.eval(pl.element().str.strip_chars())
    .list.sort()
    .list.join("-")
)

# 3) Build a route-aware segment key (avoid collisions across routes)
# seg_key = "{노선}|{canonical_section}"
seg_key_expr = pl.concat_str(
    [
        pl.col("노선").cast(pl.Utf8).str.strip_chars(),
        pl.lit("|"),
        canonical_section_expr,
    ]
)

long_lf = long_lf.with_columns(
    [
        canonical_section_expr.alias("canonical_section"),
        seg_key_expr.alias("seg_key"),
    ]
)

# 4) Create seg_idx in pure Polars:
#    - unique seg_key list
#    - stable sort
#    - row_count -> seg_idx
seg_table_lf = (
    long_lf.select(["seg_key", "노선", "canonical_section"])
    .unique()
    .sort("seg_key")
    .with_row_index("seg_idx")
    .with_columns(pl.col("seg_idx").cast(pl.Int64))
)

# Join seg_idx back to the main long_lf
long_lf = long_lf.join(
    seg_table_lf.select(["seg_key", "seg_idx"]),
    on="seg_key",
    how="left",
)

# Materialize seg_meta_df for later Excel reconstruction / debugging
seg_meta_df = seg_table_lf.collect().sort("seg_idx")
S = int(seg_meta_df.height)

# Python dict: seg_key -> seg_idx (fast lookup during Excel write-back)
SEGKEY2ID: dict[str, int] = dict(
    zip(seg_meta_df["seg_key"].to_list(), seg_meta_df["seg_idx"].to_list())
)
ID2SEGKEY: dict[int, str] = dict(
    zip(seg_meta_df["seg_idx"].to_list(), seg_meta_df["seg_key"].to_list())
)

print(f"Total physical segments S = {S}")
display(seg_meta_df.head(10))

# 5) Sort + materialize (downstream pivot + torch construction is easier in eager mode)
long_df = long_lf.sort(
    ["월", "요일타입_id", "방향_id", "seg_idx", "시간"]
).collect()

# Basic integrity checks
assert (
    long_df.select(pl.col("seg_idx").is_null().any()).item() is False
), "seg_idx has nulls!"
assert (
    long_df.select(pl.col("row_in_sheet").is_null().any()).item() is False
), "row_in_sheet has nulls!"

display(long_df.head(10))

## 5. Build X (group keys) and Y (segment × hour grid)
Compute segment count S and hours T=24. Enumerate group keys (month, weekday_id, direction_id) and build Y tensors per group shaped (S, T).


In [ ]:
# long_df is expected to contain columns:
#   (월, 요일타입_id, 방향_id, seg_idx, 시간, 지표, row_in_sheet, seg_key, canonical_section, ...)

S = int(long_df.select(pl.col("seg_idx")).n_unique())
T = 24  # hours 0-23
print("S (segment count) =", S, "T (hour count) =", T)

# Define group axes: (month, weekday_id, direction_id)
group_cols = ["월", "요일타입_id", "방향_id"]

# Stable group order
groups_df = long_df.select(group_cols).unique().sort(group_cols)

X_keys: list[tuple[int, int, int]] = [
    (int(row[0]), int(row[1]), int(row[2])) for row in groups_df.to_numpy()
]

B = len(X_keys)
print("B (group count) =", B)
print("Sample keys (first 5):", X_keys[:5])

# Convenient lookup: (month, day_id, dir_id) -> batch index
KEY2IDX: dict[tuple[int, int, int], int] = {k: i for i, k in enumerate(X_keys)}

In [ ]:
import torch


def build_Y_for_group(df_group: pl.DataFrame, S: int, T: int) -> torch.Tensor:
    """
    df_group: Polars DF containing only rows for the group (월, 요일타입_id, 방향_id)
              columns include seg_idx, 시간, 지표
    """
    # Initialize with zeros
    import numpy as np

    Y_np = np.zeros((S, T), dtype=np.float32)

    for row in df_group.iter_rows(named=True):
        s = int(row["seg_idx"])
        t = int(row["시간"])
        v = float(row["지표"])
        if 0 <= s < S and 0 <= t < T:
            Y_np[s, t] = v

    return torch.from_numpy(Y_np)  # (S, T)

In [ ]:
# Build a TensorDict dataset:
#   td_train['X']       = (month, day_id, dir_id)              shape (B, 3)
#   td_train['Y']       = (seg_idx, hour) grid                 shape (B, S, 24)
#   td_train['row_ids'] = (seg_idx -> original row_in_sheet)   shape (B, S)
#
# Why row_ids?
#   - Enables strong matching from model outputs back to the ORIGINAL Excel row positions
#   - Great for debugging alignment issues (segment mapping / duplicates / missing rows)

import numpy as np
import torch
from tensordict import TensorDict

T = 24  # hours 0-23
hour_cols = [str(h) for h in range(T)]

# ---------------------------------------------------------------------
# 1) Fast Y construction via Polars pivot (vectorized)
# ---------------------------------------------------------------------

# Complete grid (group × seg_idx) so every group has an (S, 24) matrix
seg_df = pl.DataFrame({"seg_idx": np.arange(S, dtype=np.int64)})

full_grid = groups_df.join(
    seg_df, how="cross"
)  # (B*S, 4) with columns: group_cols + seg_idx

pivot_df = long_df.pivot(
    index=group_cols + ["seg_idx"],
    columns="시간",
    values="지표",
    aggregate_function="first",
)

y_full = full_grid.join(pivot_df, on=group_cols + ["seg_idx"], how="left")

# Ensure all hour columns exist even if the dataset is missing some hours entirely
for hc in hour_cols:
    if hc not in y_full.columns:
        y_full = y_full.with_columns(pl.lit(None).alias(hc))

y_full = y_full.with_columns(
    [pl.col(hc).fill_null(0.0).cast(pl.Float32) for hc in hour_cols]
).sort(group_cols + ["seg_idx"])

Y_np = y_full.select(hour_cols).to_numpy()
assert Y_np.shape == (
    B * S,
    T,
), f"Unexpected pivot shape: {Y_np.shape} (expected {(B*S, T)})"
Y_np = Y_np.reshape((B, S, T)).astype(np.float32)

# ---------------------------------------------------------------------
# 2) Build row_ids (seg_idx -> original row_in_sheet) per group
# ---------------------------------------------------------------------

row_map = (
    long_df.select(group_cols + ["seg_idx", "row_in_sheet"])
    .group_by(group_cols + ["seg_idx"])
    .agg(pl.col("row_in_sheet").min().alias("row_in_sheet"))
)

row_full = (
    full_grid.join(row_map, on=group_cols + ["seg_idx"], how="left")
    .with_columns(pl.col("row_in_sheet").fill_null(-1).cast(pl.Int64))
    .sort(group_cols + ["seg_idx"])
)

row_ids_np = row_full.select("row_in_sheet").to_numpy()
assert row_ids_np.shape == (
    B * S,
    1,
), f"Unexpected row_ids shape: {row_ids_np.shape}"
row_ids_np = row_ids_np.reshape((B, S)).astype(np.int64)

# ---------------------------------------------------------------------
# 3) Package into TensorDict
# ---------------------------------------------------------------------

X_tensor = torch.tensor(X_keys, dtype=torch.float32)  # (B, 3)
Y_tensor = torch.from_numpy(Y_np)  # (B, S, 24)
row_ids_tensor = torch.from_numpy(row_ids_np)  # (B, S)

td_train = TensorDict(
    {"X": X_tensor, "Y": Y_tensor, "row_ids": row_ids_tensor},
    batch_size=[B],
)

print("td_train:", td_train)
print("batch_size:", td_train.batch_size)
print("X shape:", tuple(td_train["X"].shape))
print("Y shape:", tuple(td_train["Y"].shape))
print("row_ids shape:", tuple(td_train["row_ids"].shape))

# --- Sanity check: does td_train[i] behave like (x_i, y_i) and row_ids align? ---
for i in range(min(3, len(td_train))):
    xi = td_train[i]["X"]
    yi = td_train[i]["Y"]
    rid = td_train[i]["row_ids"]

    present = int((rid >= 0).sum().item())
    print(
        f"i={i}  X={xi.tolist()}  Y.shape={tuple(yi.shape)}  present_rows={present}/{S}"
    )

    # show a few mapped rows for debugging
    shown = 0
    for seg_idx in (rid >= 0).nonzero(as_tuple=False).flatten().tolist():
        if shown >= 3:
            break
        row_in_sheet = int(rid[seg_idx].item())
        seg_key = ID2SEGKEY.get(int(seg_idx), "<missing>")
        print(
            f"    seg_idx={seg_idx}  row_in_sheet={row_in_sheet}  seg_key={seg_key}"
        )
        shown += 1

## 6. Configure STNet model
Use Dataset.preprocess to infer feature and label shapes, then set PatchConfig/ModelConfig. The config picks smaller dims on CPU and larger on CUDA.


In [ ]:
from stnet.config import PatchConfig, ModelConfig
from stnet.api import new_model
from stnet.core.system import get_device
from stnet.data.pipeline import Dataset

device = get_device()
print("Device:", device)

ds = Dataset.for_device(device)
feats, labels, keys, label_shape = ds.preprocess(td_train)
# feats: (B, D)   # D = feature dimension derived from the X tuple
# labels: (B, S, T)
print("feats.shape  =", feats.shape)
print("labels.shape =", labels.shape)
print("label_shape  =", label_shape)

B2, D = feats.shape
assert label_shape == (S, T)

patch = PatchConfig(
    is_cube=True,
    grid_size_3d=(S, T, 1),
    patch_size_3d=(1, 1, 1),
    use_padding=True,
)

if device.type == "cuda":
    config = ModelConfig(
        device=device,
        patch=patch,
        normalization_method="layernorm",
        d_model=1024,
        heads=16,
        mlp_ratio=4.0,
        dropout=0.00,
        drop_path=0.00,
        spatial_depth=4,
        temporal_depth=4,
        spatial_latents=64,
        temporal_latents=64,
        modeling_type="spatiotemporal",
        compile_mode="reduce_overhead",
    )
else:
    config = ModelConfig(
        device=device,
        patch=patch,
        normalization_method="layernorm",
        d_model=256,
        heads=2,
        mlp_ratio=2.0,
        dropout=0.05,
        drop_path=0.05,
        spatial_depth=2,
        temporal_depth=2,
        spatial_latents=32,
        temporal_latents=32,
        modeling_type="spatiotemporal",
        compile_mode="disabled",
    )

model = new_model(in_dim=D, out_shape=label_shape, config=config).to(device)

## 7. Train the model
Use `train_data` as a dict mapping `(month, weekday_id, direction_id)` to a `(S, T)` torch tensor with the 24-hour grid. Pass the dict directly into `train`.


In [ ]:
import sys
from stnet.api import train

# Number of samples equals the number of dict entries
num_samples = len(td_train)
base_params = {
    "epochs": 100 if device.type == "cuda" else 50,
    "base_lr": 3e-3,
    "weight_decay": 1e-4,
}

print("num_samples:", num_samples)
print("base_params:", base_params)

# Pass td_train (Dict[X_tuple -> Y_tensor]) directly
model = train(
    model,
    td_train,
    epochs=int(base_params["epochs"]),
    base_lr=float(base_params["base_lr"]),
    weight_decay=float(base_params["weight_decay"]),
    val_frac=0.1,
)

In [ ]:
history = model.history()
for idx, item in enumerate(history):
    print(idx, item)

## 8. Run inference
Build `infer_samples` as a dict keyed by `(month, weekday_id, direction_id)` with `None` values; `predict` fills the `(S, T)` outputs and can return either in-memory or file-backed TensorDicts.


In [ ]:
import os
import torch
from tensordict import TensorDict, PersistentTensorDict

from stnet.core.system import get_device
from stnet.api import predict

device = get_device()
print("Device:", device)

# Inference input: only features (X). Labels are intentionally omitted.
infer_td = TensorDict({"X": td_train["X"]}, batch_size=td_train.batch_size)

# ---- Predict to FILE (HDF5) ----
PRED_H5 = os.path.abspath("predictions.h5")

pred_td = predict(
    model,
    infer_td,
    output="file",
    path=PRED_H5,
    overwrite="replace",
)

print("Saved predictions to:", PRED_H5)
print("pred_td:", pred_td)
print("pred_td batch_size:", pred_td.batch_size)
print("pred_td keys:", list(pred_td.keys()))

### Validate TensorDict schema (x/y)
Convert the inference result to a TensorDict with `['x', 'y']` keys to match the expected schema and confirm shapes.


In [ ]:
import os
import numpy as np
import torch
from tensordict import PersistentTensorDict

assert os.path.exists(PRED_H5), f"Missing prediction file: {PRED_H5}"

# Re-open read-only (safe if you restart the kernel later)
pred_td_ro = PersistentTensorDict(filename=PRED_H5, mode="r")

# Build a key->index map directly from the persisted X (guaranteed consistent)
X_np = pred_td_ro["X"]
if hasattr(X_np, "detach"):
    X_np = X_np.detach()
if hasattr(X_np, "cpu"):
    X_np = X_np.cpu()
X_np = np.asarray(X_np)

PRED_KEY2IDX: dict[tuple[int, int, int], int] = {
    tuple(map(int, X_np[i].tolist())): i for i in range(int(X_np.shape[0]))
}

print("pred_td_ro batch_size:", pred_td_ro.batch_size)
print("X shape:", tuple(pred_td_ro["X"].shape))
print("Y shape:", tuple(pred_td_ro["Y"].shape))
print("PRED_KEY2IDX sample:", list(PRED_KEY2IDX.items())[:3])

In [ ]:
# Debug: alignment check between prediction rows and original Excel rows via td_train['row_ids']

# Pre-build a seg_idx -> meta dict for quick lookup
SEG_META: dict[int, dict] = {
    int(r["seg_idx"]): r for r in seg_meta_df.to_dicts()
}

for i in range(min(3, len(td_train))):
    x = td_train[i]["X"]
    rid = td_train[i]["row_ids"]

    # Fetch prediction from the file-backed TensorDict (same index order as td_train)
    y_pred = pred_td_ro[i]["Y"]

    present = int((rid >= 0).sum().item())
    print(
        f"i={i} X={x.tolist()}  Y_pred.shape={tuple(y_pred.shape)}  present_rows={present}/{S}"
    )

    # Show a few rows that exist in the original sheet for this group
    segs = (rid >= 0).nonzero(as_tuple=False).flatten()[:5].tolist()
    for seg_idx in segs:
        meta = SEG_META.get(int(seg_idx), {})
        print(
            f"  seg_idx={seg_idx} row_in_sheet={int(rid[seg_idx])}  route={meta.get('노선')}  segment={meta.get('canonical_section')}"
        )

In [ ]:
import os
import numpy as np
import pandas as pd
import openpyxl
import torch

# --- Repack predictions to an Excel that mirrors the ORIGINAL workbook ---
# Requirement:
#   - same sheet names
#   - same columns (exact header row)
#   - values replaced with model predictions

HOUR_COLS = [f"{h:02d}시" for h in range(24)]


def canonicalize_section_py(sec: object) -> str:
    """Canonicalize 'A-B' and 'B-A' into the same key (same logic as canonical_section_expr)."""
    if sec is None:
        return ""
    s = str(sec).strip()
    if not s:
        return ""
    parts = [p.strip() for p in s.split("-") if p.strip()]
    if len(parts) <= 1:
        return s
    parts.sort()
    return "-".join(parts)


output_file = os.path.abspath("pred_results_like_raw.xlsx")

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    for sh in sheet_names:
        df_raw = pd.read_excel(EXCEL_PATH, sheet_name=sh)

        month, day_kind = parse_sheet_name(sh)
        day_id = DAY2ID[day_kind]

        df_out = df_raw.copy()

        # Build seg_key per original row (route-aware) without changing the output schema
        if "노선" in df_raw.columns:
            route_series = df_raw["노선"].astype(str).str.strip()
        else:
            route_series = pd.Series(
                ["수도권제1순환선"] * len(df_raw), index=df_raw.index
            )

        df_out["_seg_key"] = (
            route_series + "|" + df_raw["구간"].map(canonicalize_section_py)
        )
        df_out["_seg_idx"] = df_out["_seg_key"].map(SEGKEY2ID)

        # Fill hour columns using the right group prediction for each direction
        for dir_name, dir_id in DIR2ID.items():
            gkey = (int(month), int(day_id), int(dir_id))
            bi = PRED_KEY2IDX.get(gkey, None)  # batch index

            if bi is None:
                print(
                    f"[warn] missing predictions: group={gkey} (sheet={sh}, dir={dir_name})"
                )
                continue

            mask_dir = df_out["방향"] == dir_name
            idxs = df_out.loc[mask_dir, "_seg_idx"]
            valid = idxs.notna()
            if not bool(valid.any()):
                continue

            row_seg = idxs[valid].astype(int).to_numpy()

            # Fetch only needed rows from the file-backed tensor (avoid loading full (S,24) when possible)
            mat = pred_td_ro[int(bi)]["Y"]  # (S, 24)
            vals = mat[row_seg, :24].detach().cpu().numpy()

            df_out.loc[mask_dir & valid, HOUR_COLS] = vals

        # Drop helpers + enforce original column order
        df_out = df_out.drop(columns=["_seg_key", "_seg_idx"])
        df_out = df_out[df_raw.columns]

        df_out.to_excel(writer, sheet_name=sh, index=False)

print("Saved:", output_file)

# --- Verify: output matches input sheet + column structure ---
wb_in = openpyxl.load_workbook(EXCEL_PATH, read_only=True, data_only=True)
wb_out = openpyxl.load_workbook(output_file, read_only=True, data_only=True)

assert wb_in.sheetnames == wb_out.sheetnames, "Sheet names mismatch!"

for sh in wb_in.sheetnames:
    cols_in = next(wb_in[sh].iter_rows(min_row=1, max_row=1, values_only=True))
    cols_out = next(
        wb_out[sh].iter_rows(min_row=1, max_row=1, values_only=True)
    )
    assert list(cols_in) == list(cols_out), f"Column mismatch in sheet: {sh}"

print(
    "[ok] Verified: output Excel has the same sheets and columns as the input."
)

# Optional: close read-only prediction handle
pred_td_ro.close()